In [ ]:
import optuna
import re
from optuna.storages import RDBStorage
import pandas as pd
import seaborn as sns

pattern = r"bocl/tune_detector/([^/]+)/([^/]+)/CUSUM"
storage = RDBStorage("sqlite:///logs/optuna.db")

studies = optuna.get_all_study_names(storage)
studies = [s for s in studies if re.match(pattern, s)]

In [ ]:
study_dfs = {}
for study in studies:
    df = optuna.load_study(study_name=study, storage=storage).trials_dataframe()
    study_dfs[study] = df

df = pd.concat(study_dfs.values(), keys=study_dfs.keys())
df = df.reset_index(level=1, drop=True).reset_index().rename(columns={"index": "study"})
df

In [ ]:
# sns.scatterplot(
#     data=df,
#     x="params_drift_detector.lambda_",
#     y="value",
#     hue="study",
#     style="params_drift_detector.error_stream_type",
#     legend=True
# )
# plt.xscale("log")

In [ ]:
# Violin plot
sns.violinplot(
    data=df,
    y="value",
    hue="params_drift_detector.error_stream_type",
    # hue="study",
    split=True,
    inner="quartile",
)